# Báo cáo kết quả thực nghiệm

**Đề tài:** So sánh hiệu quả các kỹ thuật biểu diễn ảnh 2D từ dữ liệu chuỗi thời gian 1D cho bài toán phân loại bằng mạng nơ-ron tích chập nhẹ

Notebook này dùng để đọc file `summary_results.csv`, tạo bảng kết quả, vẽ biểu đồ và sinh đoạn nhận xét phục vụ cập nhật bài báo.

## Đầu vào cần có

File kết quả:

```text
results/summary_results.csv
```

## Đầu ra sẽ tạo

```text
results/table4_accuracy_f1.csv
results/table5_model_cost.csv
results/best_method_by_dataset.csv
results/figure_accuracy_by_dataset.png
results/figure_average_performance.png
results/summary_analysis.txt
```


## 1. Mục tiêu của báo cáo

Báo cáo này tập trung vào bốn nhóm kết quả chính:

1. So sánh hiệu quả phân loại giữa các phương pháp theo **Accuracy** và **Macro F1-score**.
2. Đánh giá độ nhẹ của mô hình qua số tham số, thời gian huấn luyện và thời gian suy luận nếu các cột này có trong file kết quả.
3. Xác định phương pháp tốt nhất trên từng bộ dữ liệu.
4. Tạo bảng, hình và đoạn nhận xét có thể đưa trực tiếp vào bản thảo bài báo.


In [ ]:
# ============================================================
# 1. Cài đặt và import thư viện cần thiết
# ============================================================

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

print("Python:", sys.version)
print("Thư mục hiện tại:", Path.cwd())

## 2. Kết nối Google Drive nếu chạy trên Google Colab

Nếu chạy trên máy cá nhân hoặc VS Code thì có thể bỏ qua cell này.  
Nếu chạy trên Colab, hãy chạy cell dưới đây để mount Google Drive.


In [ ]:
# ============================================================
# 2. Mount Google Drive nếu chạy trên Colab
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Đã mount Google Drive.")
except Exception as e:
    print("Không chạy trên Google Colab hoặc chưa cần mount Drive.")
    print("Thông báo:", e)

## 3. Xác định đường dẫn repo và file kết quả

Cell dưới đây tự tìm file `results/summary_results.csv` ở các vị trí thường gặp:

- Thư mục hiện tại.
- Thư mục cha nếu notebook nằm trong thư mục `notebooks`.
- Google Drive theo đường dẫn `/content/drive/MyDrive/ts2img-lightcnn`.


In [ ]:
# ============================================================
# 3. Tự động tìm repo và file summary_results.csv
# ============================================================

def find_summary_file():
    candidates = []

    cwd = Path.cwd()
    candidates.append(cwd / "results" / "summary_results.csv")
    candidates.append(cwd.parent / "results" / "summary_results.csv")

    # Một số vị trí thường gặp khi chạy trên Google Colab
    candidates.append(Path("/content/ts2img-lightcnn/results/summary_results.csv"))
    candidates.append(Path("/content/drive/MyDrive/ts2img-lightcnn/results/summary_results.csv"))
    candidates.append(Path("/content/drive/MyDrive/Colab Notebooks/ts2img-lightcnn/results/summary_results.csv"))

    for p in candidates:
        if p.exists():
            return p

    return None

SUMMARY_FILE = find_summary_file()

if SUMMARY_FILE is None:
    print("Chưa tìm thấy file summary_results.csv.")
    print("Hãy sửa thủ công biến SUMMARY_FILE ở cell dưới đây.")
else:
    print("Đã tìm thấy file:")
    print(SUMMARY_FILE)

# Nếu notebook không tự tìm thấy file, sửa dòng dưới đây theo đường dẫn thực tế:
# SUMMARY_FILE = Path("results/summary_results.csv")

if SUMMARY_FILE is None or not Path(SUMMARY_FILE).exists():
    raise FileNotFoundError(
        "Không tìm thấy summary_results.csv. "
        "Hãy kiểm tra file có nằm trong thư mục results hay chưa."
    )

SUMMARY_FILE = Path(SUMMARY_FILE)
RESULT_DIR = SUMMARY_FILE.parent
print("Thư mục kết quả:", RESULT_DIR)

## 4. Đọc và kiểm tra dữ liệu kết quả

Cell này đọc file `summary_results.csv`, hiển thị kích thước dữ liệu, danh sách cột và một số dòng đầu.


In [ ]:
# ============================================================
# 4. Đọc file summary_results.csv
# ============================================================

df_raw = pd.read_csv(SUMMARY_FILE)

print("Đọc file thành công:", SUMMARY_FILE)
print("Kích thước dữ liệu:", df_raw.shape)
print("Các cột ban đầu:")
print(df_raw.columns.tolist())

display(df_raw.head())

## 5. Chuẩn hóa tên cột

Do mỗi lần chạy thực nghiệm có thể đặt tên cột hơi khác nhau, cell dưới đây chuẩn hóa về các tên thống nhất:

```text
dataset
method
accuracy
macro_f1
params
train_time_sec
inference_time_ms
```

Trong đó, bốn cột bắt buộc là `dataset`, `method`, `accuracy`, `macro_f1`.


In [ ]:
# ============================================================
# 5. Chuẩn hóa tên cột
# ============================================================

df = df_raw.copy()
df.columns = [c.strip().lower() for c in df.columns]

rename_map = {
    # Dataset
    "dataset_name": "dataset",
    "data": "dataset",
    "datasetid": "dataset",

    # Method / model / representation
    "representation": "method",
    "input_type": "method",
    "model": "method",
    "approach": "method",
    "method_name": "method",

    # Accuracy
    "acc": "accuracy",
    "test_acc": "accuracy",
    "test_accuracy": "accuracy",
    "val_accuracy": "accuracy",

    # Macro F1
    "f1": "macro_f1",
    "f1_macro": "macro_f1",
    "macro-f1": "macro_f1",
    "macro_f1_score": "macro_f1",
    "test_f1": "macro_f1",
    "test_macro_f1": "macro_f1",

    # Parameters
    "parameters": "params",
    "trainable_params": "params",
    "num_params": "params",
    "n_params": "params",

    # Training time
    "training_time": "train_time_sec",
    "train_time": "train_time_sec",
    "training_time_sec": "train_time_sec",
    "fit_time": "train_time_sec",

    # Inference time
    "infer_time": "inference_time_ms",
    "inference_time": "inference_time_ms",
    "infer_time_ms": "inference_time_ms",
    "prediction_time_ms": "inference_time_ms"
}

df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

required_cols = ["dataset", "method", "accuracy", "macro_f1"]
missing_cols = [c for c in required_cols if c not in df.columns]

if missing_cols:
    print("Thiếu các cột bắt buộc:", missing_cols)
    print("Các cột hiện có:", df.columns.tolist())
    raise ValueError("Cần kiểm tra lại tên cột trong summary_results.csv.")

# Ép kiểu số
for col in ["accuracy", "macro_f1", "params", "train_time_sec", "inference_time_ms"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Xử lý tên phương pháp cho gọn
df["dataset"] = df["dataset"].astype(str)
df["method"] = df["method"].astype(str)

print("Các cột sau khi chuẩn hóa:")
print(df.columns.tolist())

print("\nSố bộ dữ liệu:", df["dataset"].nunique())
print("Số phương pháp:", df["method"].nunique())

display(df.head())

## 6. Tổng quan nhanh về dữ liệu kết quả

Cell này kiểm tra số lượng dòng theo từng bộ dữ liệu và từng phương pháp để phát hiện thiếu kết quả.


In [ ]:
# ============================================================
# 6. Kiểm tra tổng quan dữ liệu kết quả
# ============================================================

print("Số dòng theo từng bộ dữ liệu:")
display(df["dataset"].value_counts().sort_index().to_frame("count"))

print("Số dòng theo từng phương pháp:")
display(df["method"].value_counts().sort_index().to_frame("count"))

print("Giá trị thiếu theo cột:")
display(df.isna().sum().to_frame("missing_count"))

## 7. Bảng 4: So sánh hiệu quả phân loại

Bảng này dùng để thay cho **Bảng 4** trong bài báo.

Tên bảng đề xuất:

**Bảng 4. So sánh hiệu quả phân loại giữa các biểu diễn và mô hình**


In [ ]:
# ============================================================
# 7. Tạo Bảng 4: Accuracy / Macro F1-score
# ============================================================

df["acc_f1"] = df.apply(
    lambda r: f"{r['accuracy']:.4f} / {r['macro_f1']:.4f}",
    axis=1
)

table4 = df.pivot_table(
    index="dataset",
    columns="method",
    values="acc_f1",
    aggfunc="first"
).reset_index()

table4_file = RESULT_DIR / "table4_accuracy_f1.csv"
table4.to_csv(table4_file, index=False, encoding="utf-8-sig")

print("Đã tạo file Bảng 4:", table4_file)
display(table4)

## 8. Bảng 5: Độ nhẹ và chi phí tính toán

Bảng này dùng để thay cho **Bảng 5** trong bài báo.

Tên bảng đề xuất:

**Bảng 5. Đánh giá độ nhẹ và chi phí tính toán của các mô hình**


In [ ]:
# ============================================================
# 8. Tạo Bảng 5: Độ nhẹ và chi phí tính toán
# ============================================================

agg_dict = {
    "accuracy": "mean",
    "macro_f1": "mean"
}

if "params" in df.columns:
    agg_dict["params"] = "mean"

if "train_time_sec" in df.columns:
    agg_dict["train_time_sec"] = "mean"

if "inference_time_ms" in df.columns:
    agg_dict["inference_time_ms"] = "mean"

table5 = df.groupby("method").agg(agg_dict).reset_index()

for col in table5.columns:
    if col != "method":
        table5[col] = table5[col].round(4)

table5_file = RESULT_DIR / "table5_model_cost.csv"
table5.to_csv(table5_file, index=False, encoding="utf-8-sig")

print("Đã tạo file Bảng 5:", table5_file)
display(table5)

## 9. Phương pháp tốt nhất theo từng bộ dữ liệu

Phần này xác định phương pháp có **Macro F1-score** cao nhất trên từng bộ dữ liệu.


In [ ]:
# ============================================================
# 9. Xác định phương pháp tốt nhất theo từng bộ dữ liệu
# ============================================================

valid_df = df.dropna(subset=["macro_f1"]).copy()

best_by_dataset = valid_df.loc[
    valid_df.groupby("dataset")["macro_f1"].idxmax(),
    ["dataset", "method", "accuracy", "macro_f1"]
].reset_index(drop=True)

best_file = RESULT_DIR / "best_method_by_dataset.csv"
best_by_dataset.to_csv(best_file, index=False, encoding="utf-8-sig")

print("Đã tạo file phương pháp tốt nhất:", best_file)
display(best_by_dataset)

print("Số lần mỗi phương pháp đứng đầu:")
display(best_by_dataset["method"].value_counts().to_frame("number_of_best_results"))

## 10. Hình 2: So sánh Accuracy theo từng bộ dữ liệu

Tên hình đề xuất trong bài báo:

**Hình 2. So sánh Accuracy giữa các phương pháp trên từng bộ dữ liệu**


In [ ]:
# ============================================================
# 10. Vẽ biểu đồ Accuracy theo từng bộ dữ liệu
# ============================================================

pivot_acc = df.pivot_table(
    index="dataset",
    columns="method",
    values="accuracy",
    aggfunc="mean"
)

ax = pivot_acc.plot(kind="bar", figsize=(12, 6))

plt.title("Comparison of Accuracy by Dataset and Method")
plt.xlabel("Dataset")
plt.ylabel("Accuracy")
plt.xticks(rotation=30, ha="right")
plt.legend(title="Method", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

fig1_file = RESULT_DIR / "figure_accuracy_by_dataset.png"
plt.savefig(fig1_file, dpi=300, bbox_inches="tight")
plt.show()

print("Đã lưu hình:", fig1_file)

## 11. Hình 3: So sánh hiệu năng trung bình

Tên hình đề xuất trong bài báo:

**Hình 3. So sánh Accuracy và Macro F1-score trung bình giữa các phương pháp**


In [ ]:
# ============================================================
# 11. Vẽ biểu đồ Accuracy và Macro F1-score trung bình
# ============================================================

avg_perf = df.groupby("method")[["accuracy", "macro_f1"]].mean().reset_index()

ax = avg_perf.plot(
    x="method",
    y=["accuracy", "macro_f1"],
    kind="bar",
    figsize=(10, 5)
)

plt.title("Average Classification Performance by Method")
plt.xlabel("Method")
plt.ylabel("Score")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()

fig2_file = RESULT_DIR / "figure_average_performance.png"
plt.savefig(fig2_file, dpi=300, bbox_inches="tight")
plt.show()

print("Đã lưu hình:", fig2_file)
display(avg_perf.sort_values("macro_f1", ascending=False))

## 12. Sinh đoạn nhận xét tự động cho bài báo

Đoạn dưới đây là bản nháp. Sau khi chạy, nên đọc lại và chỉnh sửa theo văn phong học thuật của bài báo.


In [ ]:
# ============================================================
# 12. Tạo đoạn nhận xét tự động
# ============================================================

method_count = best_by_dataset["method"].value_counts()

top_method = method_count.index[0]
top_count = int(method_count.iloc[0])
total_dataset = int(best_by_dataset["dataset"].nunique())

avg_perf_sorted = avg_perf.sort_values("macro_f1", ascending=False).reset_index(drop=True)

best_avg_method = avg_perf_sorted.loc[0, "method"]
best_avg_f1 = avg_perf_sorted.loc[0, "macro_f1"]
best_avg_acc = avg_perf_sorted.loc[0, "accuracy"]

analysis_text = f"""
Kết quả thực nghiệm cho thấy phương pháp {top_method} đạt Macro F1-score cao nhất trên {top_count}/{total_dataset} bộ dữ liệu được khảo sát. 
Xét theo giá trị trung bình trên toàn bộ các bộ dữ liệu, phương pháp {best_avg_method} đạt Accuracy trung bình là {best_avg_acc:.4f} và Macro F1-score trung bình là {best_avg_f1:.4f}. 
Điều này cho thấy hiệu quả của từng kỹ thuật biểu diễn phụ thuộc vào đặc trưng của bộ dữ liệu chuỗi thời gian.

Khi một biểu diễn 2D như GAF, MTF, RP hoặc ảnh thời gian - tần số đạt kết quả cao hơn 1D-CNN, có thể nhận định rằng việc chuyển đổi chuỗi thời gian sang ảnh đã giúp mô hình 2D-CNN khai thác thêm quan hệ không gian giữa các điểm thời gian. 
Ngược lại, nếu 1D-CNN đạt kết quả tương đương hoặc tốt hơn, điều đó cho thấy dữ liệu có thể phụ thuộc nhiều hơn vào đặc trưng cục bộ theo thứ tự thời gian, và việc chuyển đổi sang ảnh 2D chưa chắc mang lại lợi ích rõ rệt.

Do đó, việc lựa chọn kỹ thuật biểu diễn cần được cân nhắc đồng thời theo ba tiêu chí: hiệu quả phân loại, độ ổn định trên nhiều bộ dữ liệu và chi phí tính toán của mô hình. 
Cách đánh giá này phù hợp với mục tiêu của nghiên cứu là tìm kiếm giải pháp cân bằng giữa độ chính xác và khả năng triển khai của mạng nơ-ron tích chập nhẹ.
""".strip()

analysis_file = RESULT_DIR / "summary_analysis.txt"
analysis_file.write_text(analysis_text, encoding="utf-8")

print(analysis_text)
print("\nĐã lưu đoạn nhận xét vào file:", analysis_file)

## 13. Gợi ý cập nhật bài báo

Sau khi chạy xong notebook này, cần cập nhật bản thảo Word theo thứ tự:

1. Thay **Bảng 4** bằng file `table4_accuracy_f1.csv`.
2. Thay **Bảng 5** bằng file `table5_model_cost.csv`.
3. Chèn hình `figure_accuracy_by_dataset.png` hoặc `figure_average_performance.png`.
4. Cập nhật phần **2.6. Cách phân tích và diễn giải kết quả** theo đoạn trong `summary_analysis.txt`.
5. Cập nhật lại **Tóm tắt**, **Abstract** và **Kết luận**, bỏ các câu nói rằng chưa có kết quả thực nghiệm.


## 14. Kiểm tra danh sách file đầu ra

Cell cuối cùng liệt kê các file đã được tạo trong thư mục `results`.


In [ ]:
# ============================================================
# 14. Kiểm tra các file đầu ra
# ============================================================

output_files = [
    "table4_accuracy_f1.csv",
    "table5_model_cost.csv",
    "best_method_by_dataset.csv",
    "figure_accuracy_by_dataset.png",
    "figure_average_performance.png",
    "summary_analysis.txt",
]

print("Các file đầu ra:")
for filename in output_files:
    p = RESULT_DIR / filename
    status = "OK" if p.exists() else "CHƯA CÓ"
    print(f"{status:8s} {p}")